In [2]:
import re
from pathlib import Path
import polars as pl

# =========================================================
# Config
# =========================================================
CATEGORY = "Pet_Supplies"

DATA_DIR  = Path("/Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data")
ITEMS_PATH = DATA_DIR / "prepared"   / f"{CATEGORY}_items_merged.parquet"
SIDS_PATH  = DATA_DIR / "embeds"     / f"{CATEGORY}_items_with_semantic_ids.parquet"
SEQS_PATH  = DATA_DIR / "sequences"  / f"{CATEGORY}_sequences_with_sid_train.parquet"

OUT_DIR = DATA_DIR / "semantic_llm_training"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = OUT_DIR / f"{CATEGORY}_conversations_train.parquet"
VAL_PATH   = OUT_DIR / f"{CATEGORY}_conversations_val.parquet"

SEED = 42
VAL_FRAC = 0.05

SYSTEM_PROMPT = """
You are a helpful AI assistant that understands and works with semantic IDs for product recommendations.

Semantic IDs are hierarchical identifiers in the format:
<|sid_start|><|Axx|><|Byy|><|Czz|><|Dww|><|sid_end|>

A/B/C are semantic codes from a quantizer; D disambiguates collisions within the same A/B/C group.
""".strip()

In [2]:
# =========================================================
# Helpers
# =========================================================
def pick_first_existing(df: pl.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"None of columns exist: {candidates}")
    return None


def ensure_text(df: pl.DataFrame, col: str) -> pl.Expr:
    # normalize nulls -> ""
    return (
        pl.when(pl.col(col).is_null())
          .then(pl.lit(""))
          .otherwise(pl.col(col).cast(pl.Utf8))
          .str.strip_chars()
          .alias(col)
    )


def build_sid_tokens_expr_from_abcd(a="A", b="B", c="C", d="D") -> pl.Expr:
    # <|sid_start|><|A0|><|B12|><|C8|><|D0|><|sid_end|>
    return pl.concat_str(
        [
            pl.lit("<|sid_start|>"),
            pl.lit("<|A"), pl.col(a).cast(pl.Utf8), pl.lit("|>"),
            pl.lit("<|B"), pl.col(b).cast(pl.Utf8), pl.lit("|>"),
            pl.lit("<|C"), pl.col(c).cast(pl.Utf8), pl.lit("|>"),
            pl.lit("<|D"), pl.col(d).cast(pl.Utf8), pl.lit("|>"),
            pl.lit("<|sid_end|>"),
        ],
        separator="",
    ).alias("sid_tokens")


def build_abcd_from_semantic_ids_list(df: pl.DataFrame, col="semantic_ids") -> pl.DataFrame:
    """
    If sids file has semantic_ids: list[i64] of length 3,
    convert it to A,B,C and set D=0 (or keep D if exists later).
    """
    if col not in df.columns:
        return df

    # Expect [A,B,C]
    return df.with_columns(
        [
            pl.col(col).list.get(0).cast(pl.Int16).alias("A"),
            pl.col(col).list.get(1).cast(pl.Int16).alias("B"),
            pl.col(col).list.get(2).cast(pl.Int16).alias("C"),
            pl.lit(0).cast(pl.Int16).alias("D"),
        ]
    )


def make_conversations_df(samples: pl.DataFrame) -> pl.DataFrame:
    # samples: instruction, output, type
    return (
        samples.select(
            [
                pl.struct([pl.lit("system").alias("role"), pl.lit(SYSTEM_PROMPT).alias("content")]).alias("s"),
                pl.struct([pl.lit("user").alias("role"), pl.col("instruction").alias("content")]).alias("u"),
                pl.struct([pl.lit("assistant").alias("role"), pl.col("output").alias("content")]).alias("a"),
                pl.col("type"),
            ]
        )
        .with_columns(pl.concat_list(["s", "u", "a"]).alias("conversations"))
        .select(["conversations", "type"])
    )


def length_report(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    rows = df.height
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        s = df.select(pl.col(c).fill_null("").cast(pl.Utf8).str.len_chars().alias("len"))["len"]
        out.append(
            {
                "col": c,
                "rows": rows,
                "max_len": int(s.max()),
                "p50": float(s.quantile(0.50)),
                "p90": float(s.quantile(0.90)),
                "p99": float(s.quantile(0.99)),
                "over_1200": int((s > 1200).sum()),
                "over_2000": int((s > 2000).sum()),
            }
        )
    return pl.DataFrame(out).sort("max_len", descending=True)

def as_text(expr: pl.Expr) -> pl.Expr:
    # гарантируем строку без null
    return expr.cast(pl.Utf8, strict=False).fill_null("")

In [3]:
# =========================================================
# Load data
# =========================================================
items = pl.read_parquet(ITEMS_PATH)
sids  = pl.read_parquet(SIDS_PATH)
seqs  = pl.read_parquet(SEQS_PATH)

asin_col = "parent_asin"

title_col = pick_first_existing(items, ["title", "original_title"], required=True)
desc_col  = pick_first_existing(items, ["description_text", "original_description"], required=False)
cat_col   = pick_first_existing(items, ["categories_text"], required=False)
feat_col  = pick_first_existing(items, ["features_text", "original_features_text"], required=False)

# --- normalize item text fields (NO trimming)
items = items.with_columns(
    [
        ensure_text(items, title_col),
        *( [ensure_text(items, desc_col)]  if desc_col  else [] ),
        *( [ensure_text(items, cat_col)]   if cat_col   else [] ),
        *( [ensure_text(items, feat_col)]  if feat_col  else [] ),
    ]
)

# --- ensure we have sid_tokens in sids
if "sid_tokens" not in sids.columns:
    # maybe you have A/B/C/D already
    if all(c in sids.columns for c in ["A", "B", "C", "D"]):
        sids = sids.with_columns(build_sid_tokens_expr_from_abcd("A", "B", "C", "D"))
    # maybe you have semantic_ids: list[i64] (len=3)
    elif "semantic_ids" in sids.columns:
        sids = build_abcd_from_semantic_ids_list(sids, "semantic_ids").with_columns(
            build_sid_tokens_expr_from_abcd("A", "B", "C", "D")
        )
    else:
        raise ValueError(
            "SIDS_PATH must contain either sid_tokens, or A/B/C/D, or semantic_ids:list[int] (len=3)."
        )

# enforce D exists as int16 (even if it was missing / built)
if "D" not in sids.columns and all(c in sids.columns for c in ["A", "B", "C"]):
    sids = sids.with_columns(pl.lit(0).cast(pl.Int16).alias("D"))

sid_token_col = "sid_tokens"

# Join: items + sid_tokens (+ A/B/C/D if present)
keep_sid_cols = [asin_col, "sid_tokens"] + [c for c in ["A", "B", "C", "D"] if c in sids.columns]
df = items.join(sids.select(keep_sid_cols), on=asin_col, how="inner")

print("Joined df:", df.shape)
print("Text length report (no trimming):")
print(length_report(df, ["title", "description_text", "categories_text", "features_text", "sid_tokens"]))

# Extract 2nd-level subcategory: "Pet Supplies > Dogs > ..." -> "Dogs"
# main_category == "Pet Supplies" for ALL items -> useless for discrimination
if cat_col and cat_col in df.columns:
    df = df.with_columns(
        pl.col(cat_col)
          .str.split(" > ")
          .list.get(1, null_on_oob=True)
          .fill_null(pl.col(cat_col).str.split(" > ").list.get(0, null_on_oob=True))
          .alias("subcat_2nd")
    )
    print("subcat_2nd unique:", df["subcat_2nd"].n_unique())
    print(df["subcat_2nd"].value_counts().sort("count", descending=True).head(10))

Joined df: (63319, 24)
Text length report (no trimming):
shape: (5, 8)
┌──────────────────┬───────┬─────────┬───────┬───────┬────────┬───────────┬───────────┐
│ col              ┆ rows  ┆ max_len ┆ p50   ┆ p90   ┆ p99    ┆ over_1200 ┆ over_2000 │
│ ---              ┆ ---   ┆ ---     ┆ ---   ┆ ---   ┆ ---    ┆ ---       ┆ ---       │
│ str              ┆ i64   ┆ i64     ┆ f64   ┆ f64   ┆ f64    ┆ i64       ┆ i64       │
╞══════════════════╪═══════╪═════════╪═══════╪═══════╪════════╪═══════════╪═══════════╡
│ description_text ┆ 63319 ┆ 2005    ┆ 591.0 ┆ 900.0 ┆ 1272.0 ┆ 956       ┆ 1         │
│ features_text    ┆ 63319 ┆ 1676    ┆ 344.0 ┆ 512.0 ┆ 680.0  ┆ 14        ┆ 0         │
│ title            ┆ 63319 ┆ 215     ┆ 69.0  ┆ 118.0 ┆ 176.0  ┆ 0         ┆ 0         │
│ categories_text  ┆ 63319 ┆ 97      ┆ 56.0  ┆ 80.0  ┆ 91.0   ┆ 0         ┆ 0         │
│ sid_tokens       ┆ 63319 ┆ 55      ┆ 53.0  ┆ 54.0  ┆ 54.0   ┆ 0         ┆ 0         │
└──────────────────┴───────┴─────────┴───────┴───

In [ ]:
# =========================================================
# Type A: SID -> Text (vectorized, no trimming)  ✅ OK
# =========================================================
a_blocks = []

a_blocks.append(
    df.select([
        pl.concat_str([pl.lit("Product "), as_text(pl.col(sid_token_col)), pl.lit(" has title:")]).alias("instruction"),
        as_text(pl.col(title_col)).alias("output"),
        pl.lit("sid_to_title").alias("type"),
    ])
)

if desc_col:
    a_blocks.append(
        df.filter(as_text(pl.col(desc_col)).str.len_chars() >= 1).select([
            pl.concat_str([pl.lit("Item "), as_text(pl.col(sid_token_col)), pl.lit(" is described as:")]).alias("instruction"),
            as_text(pl.col(desc_col)).alias("output"),
            pl.lit("sid_to_description").alias("type"),
        ])
    )

if cat_col:
    a_blocks.append(
        df.filter(as_text(pl.col(cat_col)).str.len_chars() >= 1).select([
            pl.concat_str([pl.lit("Product "), as_text(pl.col(sid_token_col)), pl.lit(" belongs to category:")]).alias("instruction"),
            as_text(pl.col(cat_col)).alias("output"),
            pl.lit("sid_to_category").alias("type"),
        ])
    )

context_parts = [pl.lit("Title: "), as_text(pl.col(title_col))]
if desc_col:
    context_parts += [pl.lit(" Description: "), as_text(pl.col(desc_col))]
if cat_col:
    context_parts += [pl.lit(" Category: "), as_text(pl.col(cat_col))]

a_blocks.append(
    df.select([
        pl.concat_str([pl.lit("Tell me about product "), as_text(pl.col(sid_token_col)), pl.lit(":")]).alias("instruction"),
        pl.concat_str(context_parts).alias("output"),
        pl.lit("sid_to_context").alias("type"),
    ])
)

if feat_col:
    a_blocks.append(
        df.filter(as_text(pl.col(feat_col)).str.len_chars() >= 1).select([
            pl.concat_str([pl.lit("Product "), as_text(pl.col(sid_token_col)), pl.lit(" features:")]).alias("instruction"),
            as_text(pl.col(feat_col)).alias("output"),
            pl.lit("sid_to_features").alias("type"),
        ])
    )

type_a = pl.concat(a_blocks, how="vertical")


# =========================================================
# Type B: Text -> SID (vectorized, deterministic) ✅ + ADD ALL MISSING
# Adds:
# - title_prefix_to_sid
# - description_contains_to_sid
# =========================================================
b_blocks = []

# title_to_sid
b_blocks.append(
    df.select([
        pl.concat_str([pl.lit('The product "'), as_text(pl.col(title_col)), pl.lit('" has SemanticID:')]).alias("instruction"),
        as_text(pl.col(sid_token_col)).alias("output"),
        pl.lit("title_to_sid").alias("type"),
    ])
)

# title_prefix_to_sid (prefix = first 6 words; only if title has >=3 words)
b_blocks.append(
    df.with_columns([
        as_text(pl.col(title_col)).str.split(" ").alias("_tw"),
        as_text(pl.col(title_col)).str.split(" ").list.len().alias("_tw_len"),
    ])
    .filter(pl.col("_tw_len") >= 3)
    .with_columns([
        pl.col("_tw").list.slice(0, 6).list.join(" ").alias("_pref"),
    ])
    .select([
        pl.concat_str([
            pl.lit('A product titled "'),
            as_text(pl.col("_pref")),
            pl.lit('" has SemanticID:')
        ]).alias("instruction"),
        as_text(pl.col(sid_token_col)).alias("output"),
        pl.lit("title_prefix_to_sid").alias("type"),
    ])
)

# title_contains_to_sid: deterministic middle 4-gram (Polars-safe)
b_blocks.append(
    df.with_columns([
        as_text(pl.col(title_col)).str.split(" ").alias("_tw"),
        as_text(pl.col(title_col)).str.split(" ").list.len().alias("_tw_len"),
    ])
    .filter(pl.col("_tw_len") >= 8)
    .with_columns([
        pl.max_horizontal([pl.col("_tw_len") // 2 - 2, pl.lit(0)]).alias("_i0"),
    ])
    .with_columns([
        pl.col("_tw").list.slice(pl.col("_i0"), 4).list.join(" ").alias("_mid"),
    ])
    .select([
        pl.concat_str([
            pl.lit('A product with "...'),
            as_text(pl.col("_mid")),
            pl.lit('..." in its title has SemanticID:')
        ]).alias("instruction"),
        as_text(pl.col(sid_token_col)).alias("output"),
        pl.lit("title_contains_to_sid").alias("type"),
    ])
)

# description_to_sid
if desc_col:
    b_blocks.append(
        df.filter(as_text(pl.col(desc_col)).str.len_chars() >= 50).select([
            pl.concat_str([
                pl.lit('A product described as "'),
                as_text(pl.col(desc_col)).str.slice(0, 250),
                pl.lit('..." has SemanticID:')
            ]).alias("instruction"),
            as_text(pl.col(sid_token_col)).alias("output"),
            pl.lit("description_to_sid").alias("type"),
        ])
    )

# description_contains_to_sid (offset snippets 50/100/200, len 50)
if desc_col:
    snippets_dtype = pl.List(pl.Struct([pl.Field("snippet", pl.Utf8)]))
    b_blocks.append(
        df.filter(as_text(pl.col(desc_col)).str.len_chars() >= 120)
          .with_columns([
              pl.col(desc_col).map_elements(
                  lambda s: (
                      [] if s is None else
                      [{"snippet": s[i:i+50]} for i in (50, 100, 200)
                       if len(s) >= i + 60 and len(s[i:i+50].strip()) >= 10]
                  ),
                  return_dtype=snippets_dtype,
              ).alias("_snips")
          ])
          .explode("_snips")
          .with_columns([
              pl.col("_snips").struct.field("snippet").alias("_snippet"),
          ])
          .select([
              pl.concat_str([
                  pl.lit('A product with description containing "...'),
                  as_text(pl.col("_snippet")),
                  pl.lit('..." has SemanticID:')
              ]).alias("instruction"),
              as_text(pl.col(sid_token_col)).alias("output"),
              pl.lit("description_contains_to_sid").alias("type"),
          ])
    )

# features_to_sid
if feat_col:
    b_blocks.append(
        df.filter(as_text(pl.col(feat_col)).str.len_chars() >= 20).select([
            pl.concat_str([
                pl.lit("A "),
                pl.lit(CATEGORY),
                pl.lit(' product with features "'),
                as_text(pl.col(feat_col)).str.slice(0, 400),
                pl.lit('" has SemanticID:')
            ]).alias("instruction"),
            as_text(pl.col(sid_token_col)).alias("output"),
            pl.lit("features_to_sid").alias("type"),
        ])
    )

type_b = pl.concat(b_blocks, how="vertical")


# =========================================================
# Type C: Sequence prediction (ALL split points) — Polars-safe ✅ + ADD ALL MISSING
# Adds:
# - seq_last_3
# - seq_last_5
# =========================================================
sid_seq_col = pick_first_existing(seqs, ["sid_sequence", "semantic_id_sequence"], required=True)
sid_len_col = pick_first_existing(seqs, ["sid_sequence_length", "semantic_id_sequence_length"], required=False)

seqs2 = (
    seqs.with_columns([
        (pl.col(sid_seq_col).list.len().alias("_L") if sid_len_col is None else pl.col(sid_len_col).alias("_L"))
    ])
    .filter(pl.col("_L") >= 3)
    .with_columns([
        pl.col("_L").map_elements(
            lambda L: list(range(2, int(L))),  # split points: 2..L-1
            return_dtype=pl.List(pl.Int64),
        ).alias("_splits")
    ])
)

# Base: build hist/target once
seq_base = (
    seqs2
    .explode("_splits")
    .with_columns([
        pl.col(sid_seq_col).list.slice(0, pl.col("_splits")).alias("_hist"),
        pl.col(sid_seq_col).list.get(pl.col("_splits")).alias("_target"),
        pl.col(sid_seq_col).list.len().alias("_seq_len"),
    ])
    .with_columns([
        pl.col("_hist").map_elements(lambda h: len(h) if h is not None else 0, return_dtype=pl.Int64).alias("_hL"),
    ])
)

# seq_last_2
type_c_2 = (
    seq_base
    .with_columns([
        pl.col("_hist").map_elements(lambda h: h[-2] if h is not None and len(h) >= 2 else "", return_dtype=pl.Utf8).alias("_last2"),
        pl.col("_hist").map_elements(lambda h: h[-1] if h is not None and len(h) >= 1 else "", return_dtype=pl.Utf8).alias("_last1"),
    ])
    .filter((pl.col("_last2").str.len_chars() > 0) & (pl.col("_last1").str.len_chars() > 0))
    .select([
        pl.concat_str([
            pl.lit("User's last purchases: "),
            as_text(pl.col("_last2")),
            pl.lit(", "),
            as_text(pl.col("_last1")),
            pl.lit(". Next:")
        ]).alias("instruction"),
        as_text(pl.col("_target")).alias("output"),
        pl.lit("seq_last_2").alias("type"),
    ])
)

# seq_last_3
type_c_3 = (
    seq_base
    .filter(pl.col("_hL") >= 3)
    .with_columns([
        pl.col("_hist").map_elements(
            lambda h: ", ".join(h[-3:]) if h is not None and len(h) >= 3 else "",
            return_dtype=pl.Utf8
        ).alias("_last3"),
    ])
    .filter(pl.col("_last3").str.len_chars() > 0)
    .select([
        pl.concat_str([
            pl.lit("Based on recent purchases: "),
            as_text(pl.col("_last3")),
            pl.lit(", next item:")
        ]).alias("instruction"),
        as_text(pl.col("_target")).alias("output"),
        pl.lit("seq_last_3").alias("type"),
    ])
)

# seq_last_5
type_c_5 = (
    seq_base
    .filter(pl.col("_hL") >= 5)
    .with_columns([
        pl.col("_hist").map_elements(
            lambda h: ", ".join(h[-5:]) if h is not None and len(h) >= 5 else "",
            return_dtype=pl.Utf8
        ).alias("_last5"),
    ])
    .filter(pl.col("_last5").str.len_chars() > 0)
    .select([
        pl.concat_str([
            pl.lit("Purchase sequence: "),
            as_text(pl.col("_last5")),
            pl.lit(". Predict next:")
        ]).alias("instruction"),
        as_text(pl.col("_target")).alias("output"),
        pl.lit("seq_last_5").alias("type"),
    ])
)

type_c = pl.concat([type_c_2, type_c_3, type_c_5], how="vertical")


# =========================================================
# Type D1: Similar products via same (A,B) group ✅ (yours)
# =========================================================
if all(c in df.columns for c in ["A", "B"]):
    grp = (
        df.group_by(["A", "B"])
          .agg([
              as_text(pl.col(sid_token_col)).alias("sids"),  # list[str]
              pl.len().alias("n"),
          ])
          .filter(pl.col("n") >= 2)
    )

    type_d_similar = (
        grp.with_columns([
            pl.col("sids").list.slice(0, 6).alias("_base_list")
        ])
        .explode("_base_list")
        .with_columns([
            pl.col("_base_list").alias("_base"),
            pl.struct(["sids", "_base_list"]).map_elements(
                lambda x: ", ".join([s for s in x["sids"] if s != x["_base_list"]][:5]),
                return_dtype=pl.Utf8
            ).alias("output"),
        ])
        .with_columns([
            pl.concat_str([
                pl.lit("Products similar to "),
                as_text(pl.col("_base")),
                pl.lit(" include:")
            ]).alias("instruction"),
            pl.lit("similar_products_ab").alias("type"),
        ])
        .filter(as_text(pl.col("output")).str.len_chars() > 0)
        .select(["instruction", "output", "type"])
    )
else:
    type_d_similar = pl.DataFrame({"instruction": [], "output": [], "type": []})


# =========================================================
# Type D2: Prefix/category understanding ✅ ADD ALL MISSING
# Adds:
# - prefix_category
# - prefix_examples
# - prefix_2level_category
# (uses A and A+B as "prefix")
# =========================================================
d_blocks = [type_d_similar]

if all(c in df.columns for c in ["A"]) and "subcat_2nd" in df.columns:
    prefix0 = (
        df.group_by(["A"])
          .agg([
              pl.col("subcat_2nd").mode().first().alias("common_category"),
              pl.col("subcat_2nd").n_unique().alias("n_categories"),
              pl.col(sid_token_col).alias("example_ids"),
              pl.len().alias("count"),
          ])
          .filter(pl.col("count") >= 5)
    )

    # prefix_category
    d_blocks.append(
        prefix0.select([
            pl.concat_str([
                pl.lit("Products starting with prefix A="),
                as_text(pl.col("A")),
                pl.lit(" are typically:")
            ]).alias("instruction"),
            pl.concat_str([as_text(pl.col("common_category")), pl.lit(" products")]).alias("output"),
            pl.lit("prefix_category").alias("type"),
        ])
    )

    # prefix_examples (3 examples)
    d_blocks.append(
        prefix0.with_columns([
            pl.col("example_ids").list.slice(0, 3).list.join(", ").alias("_ex3")
        ]).select([
            pl.concat_str([
                pl.lit("Examples of products with prefix A="),
                as_text(pl.col("A")),
                pl.lit(":")
            ]).alias("instruction"),
            as_text(pl.col("_ex3")).alias("output"),
            pl.lit("prefix_examples").alias("type"),
        ])
    )

if all(c in df.columns for c in ["A", "B"]) and "subcat_2nd" in df.columns:
    prefix01 = (
        df.group_by(["A", "B"])
          .agg([
              pl.col("subcat_2nd").mode().first().alias("common_category"),
              pl.len().alias("count"),
          ])
          .filter(pl.col("count") >= 3)
    )

    d_blocks.append(
        prefix01.select([
            pl.concat_str([
                pl.lit("Products starting with prefix A="),
                as_text(pl.col("A")),
                pl.lit(", B="),
                as_text(pl.col("B")),
                pl.lit(" are:")
            ]).alias("instruction"),
            pl.concat_str([as_text(pl.col("common_category")), pl.lit(" items")]).alias("output"),
            pl.lit("prefix_2level_category").alias("type"),
        ])
    )

type_d = pl.concat(d_blocks, how="vertical")


# =========================================================
# Type E: Co-purchase (forward + backward) ✅ ADD ALL MISSING
# Adds:
# - copurchase_backward
# =========================================================
pairs_dtype = pl.List(
    pl.Struct([pl.Field("sid_i", pl.Utf8), pl.Field("sid_j", pl.Utf8)])
)

pairs = (
    seqs2.select([pl.col(sid_seq_col).alias("seq")])
    .filter(pl.col("seq").is_not_null())
    .with_columns([
        pl.col("seq").map_elements(
            lambda seq: (
                [] if seq is None or len(seq) < 2 else
                [{"sid_i": seq[i], "sid_j": seq[j]}
                 for i in range(len(seq) - 1)
                 for j in range(i + 1, min(i + 5, len(seq)))]  # window=4 (как у Eugene Yan)
            ),
            return_dtype=pairs_dtype,
        ).alias("_pairs")
    ])
    .explode("_pairs")
    .with_columns([
        pl.col("_pairs").struct.field("sid_i").alias("sid_i"),
        pl.col("_pairs").struct.field("sid_j").alias("sid_j"),
    ])
    .select(["sid_i", "sid_j"])
    .unique()
)

# Строим lookup sid -> title для обогащения copurchase-инструкций (ключевое отличие Eugene Yan).
# У него в инструкции: "A user who bought <SID> (Product Title) might also buy:"
# Это делает каждый copurchase-пример двойным: паттерн покупок + связь SID↔текст.
_sid_title = (
    df.select([pl.col(sid_token_col).alias("sid"), pl.col(title_col).alias("title")])
    .unique(subset=["sid"])
)

pairs_with_titles = (
    pairs
    .join(_sid_title, left_on="sid_i", right_on="sid", how="left")
    .rename({"title": "title_i"})
    .join(_sid_title, left_on="sid_j", right_on="sid", how="left")
    .rename({"title": "title_j"})
)

type_e_forward = (
    pairs_with_titles.select([
        pl.concat_str([
            pl.lit("A user who bought "),
            as_text(pl.col("sid_i")),
            pl.lit(" ("),
            pl.col("title_i").fill_null(""),
            pl.lit(") might also buy:")
        ]).alias("instruction"),
        as_text(pl.col("sid_j")).alias("output"),
        pl.lit("copurchase_forward").alias("type"),
    ])
)

type_e_backward = (
    pairs_with_titles.select([
        pl.concat_str([
            pl.lit("Before buying "),
            as_text(pl.col("sid_j")),
            pl.lit(" ("),
            pl.col("title_j").fill_null(""),
            pl.lit("), users often buy:")
        ]).alias("instruction"),
        as_text(pl.col("sid_i")).alias("output"),
        pl.lit("copurchase_backward").alias("type"),
    ])
)

type_e = pl.concat([type_e_forward, type_e_backward], how="vertical")


# =========================================================
# Type F: category_transition + frequently_bought_together + sequence_length_example ✅ ADD ALL MISSING
# =========================================================
f_blocks = []

# ---------- category_transition ----------
# Map sid -> subcat_2nd (2nd-level: Dogs, Cats, ...)
if "subcat_2nd" in df.columns:
    sid2cat = df.select([pl.col(sid_token_col).alias("sid"), pl.col("subcat_2nd").cast(pl.Utf8).alias("cat")]).unique()

    # explode sequence into (rid, pos, sid)
    pos_dtype = pl.List(pl.Struct([pl.Field("pos", pl.Int64), pl.Field("sid", pl.Utf8)]))
    seq_pos = (
        seqs2.select([pl.col(sid_seq_col).alias("seq")])
        .with_row_index("rid")
        .with_columns([
            pl.col("seq").map_elements(
                lambda seq: (
                    [] if seq is None else [{"pos": i, "sid": seq[i]} for i in range(len(seq))]
                ),
                return_dtype=pos_dtype,
            ).alias("_pos")
        ])
        .explode("_pos")
        .with_columns([
            pl.col("_pos").struct.field("pos").alias("pos"),
            pl.col("_pos").struct.field("sid").alias("sid"),
        ])
        .select(["rid", "pos", "sid"])
        .join(sid2cat, left_on="sid", right_on="sid", how="left")
        .rename({"cat": "cat_i"})
        .sort(["rid", "pos"])
        .with_columns([
            pl.col("cat_i").shift(-1).over("rid").alias("cat_j")
        ])
        .filter(pl.col("cat_i").is_not_null() & pl.col("cat_j").is_not_null())
        .select(["cat_i", "cat_j"])
    )

    transitions = (
        seq_pos.group_by(["cat_i", "cat_j"])
               .len()
               .filter(pl.col("len") >= 5)
    )

    f_blocks.append(
        transitions.select([
            pl.concat_str([
                pl.lit("After buying from "),
                as_text(pl.col("cat_i")),
                pl.lit(", users often buy from:")
            ]).alias("instruction"),
            as_text(pl.col("cat_j")).alias("output"),
            pl.lit("category_transition").alias("type"),
        ])
    )

# ---------- frequently_bought_together ----------
# Pair counts within small local window (j <= i+2), like "min(i+3, ...)" from ref.
local_pairs_dtype = pl.List(pl.Struct([pl.Field("sid_i", pl.Utf8), pl.Field("sid_j", pl.Utf8)]))
local_pairs = (
    seqs2.select([pl.col(sid_seq_col).alias("seq")])
    .filter(pl.col("seq").is_not_null())
    .with_columns([
        pl.col("seq").map_elements(
            lambda seq: (
                [] if seq is None or len(seq) < 2 else
                [{"sid_i": seq[i], "sid_j": seq[j]}
                 for i in range(len(seq) - 1)
                 for j in range(i + 1, min(i + 3, len(seq)))]  # window=2 ahead
            ),
            return_dtype=local_pairs_dtype,
        ).alias("_lpairs")
    ])
    .explode("_lpairs")
    .select([
        pl.col("_lpairs").struct.field("sid_i").alias("sid_i"),
        pl.col("_lpairs").struct.field("sid_j").alias("sid_j"),
    ])
)

pair_counts = (
    local_pairs.group_by(["sid_i", "sid_j"]).len()
    .filter(pl.col("len") >= 3)
    .sort("len", descending=True)
    .head(500)
)

f_blocks.append(
    pair_counts.select([
        pl.concat_str([
            pl.lit("Items frequently bought together: "),
            as_text(pl.col("sid_i")),
            pl.lit(" and:")
        ]).alias("instruction"),
        as_text(pl.col("sid_j")).alias("output"),
        pl.lit("frequently_bought_together").alias("type"),
    ])
)

# ---------- sequence_length_example ----------
for L in [3, 5, 10, 20]:
    ex = (
        seqs2.filter(pl.col("_L") >= L)
        .select([
            pl.lit(f"Example of a {L}+ item purchase sequence:").alias("instruction"),
            pl.col(sid_seq_col).list.slice(0, L).list.join(", ").alias("output"),
            pl.lit("sequence_length_example").alias("type"),
        ])
        .head(10)
    )
    f_blocks.append(ex)

type_f = pl.concat(f_blocks, how="vertical") if len(f_blocks) else pl.DataFrame({"instruction": [], "output": [], "type": []})


# =========================================================
# Combine + conversation format + split + save ✅
# =========================================================
samples = (
    pl.concat([type_a, type_b, type_c, type_d, type_e, type_f], how="vertical")
    .filter(
        (as_text(pl.col("instruction")).str.len_chars() > 0) &
        (as_text(pl.col("output")).str.len_chars() > 0)
    )
)

conv = make_conversations_df(samples).sample(fraction=1.0, shuffle=True, seed=SEED)

n = conv.height
n_val = int(n * VAL_FRAC)

val = conv.head(n_val)
train = conv.slice(n_val, n - n_val)

train.write_parquet(TRAIN_PATH)
val.write_parquet(VAL_PATH)

print("Saved:")
print(" train:", TRAIN_PATH, "rows:", train.height)
print(" val  :", VAL_PATH, "rows:", val.height)

print("\nTop-20 types (train):")
print(train.group_by("type").len().sort("len", descending=True).head(20))

Saved:
 train: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/semantic_llm_training/Pet_Supplies_conversations_train.parquet rows: 4719994
 val  : /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/semantic_llm_training/Pet_Supplies_conversations_val.parquet rows: 248420

Top-20 types (train):
shape: (20, 2)
┌────────────────────────────┬─────────┐
│ type                       ┆ len     │
│ ---                        ┆ ---     │
│ str                        ┆ u32     │
╞════════════════════════════╪═════════╡
│ copurchase_forward         ┆ 1397809 │
│ copurchase_backward        ┆ 1397440 │
│ seq_last_2                 ┆ 483749  │
│ seq_last_3                 ┆ 396525  │
│ seq_last_5                 ┆ 222391  │
│ …                          ┆ …       │
│ title_contains_to_sid      ┆ 51644   │
│ similar_products_ab        ┆ 41365   │
│ prefix_2level_category     ┆ 7084    │
│ frequently_bought_together

In [5]:
import polars as pl

train = pl.read_parquet(TRAIN_PATH)
val   = pl.read_parquet(VAL_PATH)

# user+assistant ключ
train_pairs = train.select([
    pl.col("type"),
    pl.col("conversations").list.get(1).struct.field("content").alias("user"),
    pl.col("conversations").list.get(2).struct.field("content").alias("assistant"),
]).unique()

val_pairs = val.select([
    pl.col("type"),
    pl.col("conversations").list.get(1).struct.field("content").alias("user"),
    pl.col("conversations").list.get(2).struct.field("content").alias("assistant"),
]).unique()

# пересечение по (type, user, assistant)
overlap = (
    val_pairs.join(
        train_pairs,
        on=["type", "user", "assistant"],
        how="inner"
    )
)

report = (
    overlap.group_by("type").len()
    .sort("len", descending=True)
)

print("OVERLAP BY TYPE")
print(report)

print("\nTOTAL overlap:", overlap.height, "/", val_pairs.height,
      "=", overlap.height / val_pairs.height)

OVERLAP BY TYPE
shape: (3, 2)
┌────────────┬─────┐
│ type       ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ seq_last_2 ┆ 106 │
│ seq_last_3 ┆ 15  │
│ seq_last_5 ┆ 10  │
└────────────┴─────┘

TOTAL overlap: 131 / 248410 = 0.0005273539712571958


In [6]:
train.filter(pl.col("type") == "frequently_bought_together")['conversations'][0]

""
struct[2]
"{""system"",""You are a helpful AI assistant that understands and works with semantic IDs for product recommendations. Semantic IDs are hierarchical identifiers in the format: <|sid_start|><|Axx|><|Byy|><|Czz|><|Dww|><|sid_end|> A/B/C are semantic codes from a quantizer; D disambiguates collisions within the same A/B/C group.""}"
"{""user"",""Items frequently bought together: <|sid_start|><|A7|><|B78|><|C247|><|D0|><|sid_end|> and:""}"
"{""assistant"",""<|sid_start|><|A7|><|B121|><|C25|><|D0|><|sid_end|>""}"


In [7]:
import polars as pl

# Paths
TRAIN_PATH = TRAIN_PATH  # already defined in your notebook
VAL_PATH   = VAL_PATH

def add_analysis_cols(df: pl.DataFrame) -> pl.DataFrame:
    """
    Expects schema:
      - conversations: list[struct{role:str, content:str}]
      - type: str (optional but you have it)
    Adds:
      - user_message, assistant_message
      - token counts (simple whitespace split)
      - char lengths
    """
    return (
        df.with_columns([
            pl.col("conversations").list.get(1).struct.field("content").alias("user_message"),
            pl.col("conversations").list.get(2).struct.field("content").alias("assistant_message"),
        ])
        .with_columns([
            pl.col("user_message").fill_null("").cast(pl.Utf8),
            pl.col("assistant_message").fill_null("").cast(pl.Utf8),
        ])
        .with_columns([
            pl.col("user_message").str.len_chars().alias("user_message_length"),
            pl.col("assistant_message").str.len_chars().alias("assistant_message_length"),
            pl.col("user_message").str.split(" ").list.len().alias("user_message_tokens"),
            pl.col("assistant_message").str.split(" ").list.len().alias("assistant_message_tokens"),
        ])
    )

def print_final_stats(df: pl.DataFrame, name: str = "DATASET"):
    n = df.height
    print(f"\nFINAL STATISTICS (Conversation Format) — {name}")
    print("=" * 50)
    print(f"Total unique samples: {n:,}")

    # --- Data type distribution
    if "type" in df.columns:
        print("Data type distribution:")
        type_counts = df.group_by("type").len().sort("len", descending=True)
        for t, cnt in type_counts.iter_rows():
            pct = 100.0 * cnt / n if n else 0.0
            print(f"  {t}: {cnt:,} samples ({pct:.1f}%)")
    else:
        print("Data type distribution: (no `type` column)")

    # --- Token stats
    avg_u = float(df["user_message_tokens"].mean()) if n else 0.0
    avg_a = float(df["assistant_message_tokens"].mean()) if n else 0.0
    sum_tokens = int((df["user_message_tokens"] + df["assistant_message_tokens"]).sum()) if n else 0

    print("\nToken statistics:")
    print(f"  Average user message: {avg_u:.1f} tokens")
    print(f"  Average assistant message: {avg_a:.1f} tokens")
    print(f"  Average total per conversation: {(avg_u + avg_a):.1f} tokens")
    print(f"  Estimated total tokens: {sum_tokens:,}")

    # --- Diversity
    uniq_u = int(df["user_message"].n_unique()) if n else 0
    uniq_a = int(df["assistant_message"].n_unique()) if n else 0

    print("\nDiversity metrics:")
    print(f"  Unique user messages: {uniq_u:,} ({(100.0 * uniq_u / n if n else 0.0):.1f}%)")
    print(f"  Unique assistant messages: {uniq_a:,} ({(100.0 * uniq_a / n if n else 0.0):.1f}%)")

    # --- Reminder
    print("\nConversation format:")
    print("  Each sample contains a 3-turn conversation:")
    print("  1. System prompt (defines assistant role)")
    print("  2. User message (the instruction/question)")
    print("  3. Assistant response (the answer/output)")

# =========================
# Load + analyze train/val
# =========================
train_df = pl.read_parquet(TRAIN_PATH)
val_df   = pl.read_parquet(VAL_PATH)

train_df = add_analysis_cols(train_df)
val_df   = add_analysis_cols(val_df)

print_final_stats(train_df, name="TRAIN")
print(f"\nOutput files:\n  - {str(TRAIN_PATH)}")
print(f"  - {str(VAL_PATH)}")

print_final_stats(val_df, name="VAL")


FINAL STATISTICS (Conversation Format) — TRAIN
Total unique samples: 4,719,994
Data type distribution:
  copurchase_forward: 1,397,809 samples (29.6%)
  copurchase_backward: 1,397,440 samples (29.6%)
  seq_last_2: 483,749 samples (10.2%)
  seq_last_3: 396,525 samples (8.4%)
  seq_last_5: 222,391 samples (4.7%)
  description_contains_to_sid: 179,813 samples (3.8%)
  sid_to_description: 60,222 samples (1.3%)
  description_to_sid: 60,206 samples (1.3%)
  sid_to_title: 60,148 samples (1.3%)
  sid_to_category: 60,123 samples (1.3%)
  title_to_sid: 60,093 samples (1.3%)
  features_to_sid: 60,078 samples (1.3%)
  sid_to_context: 60,048 samples (1.3%)
  title_prefix_to_sid: 60,045 samples (1.3%)
  sid_to_features: 60,009 samples (1.3%)
  title_contains_to_sid: 51,644 samples (1.1%)
  similar_products_ab: 41,365 samples (0.9%)
  prefix_2level_category: 7,084 samples (0.2%)
  frequently_bought_together: 474 samples (0.0%)
  prefix_category: 239 samples (0.0%)
  prefix_examples: 236 samples (0.0

In [3]:
import pandas as pd
df = pd.read_parquet(VAL_PATH)

In [4]:
df.type.unique()


<ArrowStringArray>
[         'copurchase_forward',                  'seq_last_3',
         'copurchase_backward', 'description_contains_to_sid',
                'sid_to_title',                  'seq_last_5',
                  'seq_last_2',         'title_prefix_to_sid',
          'sid_to_description',          'description_to_sid',
       'title_contains_to_sid',                'title_to_sid',
             'sid_to_features',             'features_to_sid',
         'similar_products_ab',      'prefix_2level_category',
             'sid_to_category',              'sid_to_context',
             'prefix_examples',  'frequently_bought_together',
             'prefix_category',         'category_transition']
Length: 22, dtype: str

In [15]:
df[df['type'] == 'prefix_2level_category'].reset_index().conversations[10]

array([{'role': 'system', 'content': 'You are a helpful AI assistant that understands and works with semantic IDs for product recommendations.\n\nSemantic IDs are hierarchical identifiers in the format:\n<|sid_start|><|Axx|><|Byy|><|Czz|><|Dww|><|sid_end|>\n\nA/B/C are semantic codes from a quantizer; D disambiguates collisions within the same A/B/C group.'},
       {'role': 'user', 'content': 'Products starting with prefix A=233, B=53 are:'},
       {'role': 'assistant', 'content': 'Fish & Aquatic Pets items'}],
      dtype=object)

In [12]:
df[df['type'] == 'category_transition'].reset_index().conversations[6]

array([{'role': 'system', 'content': 'You are a helpful AI assistant that understands and works with semantic IDs for product recommendations.\n\nSemantic IDs are hierarchical identifiers in the format:\n<|sid_start|><|Axx|><|Byy|><|Czz|><|Dww|><|sid_end|>\n\nA/B/C are semantic codes from a quantizer; D disambiguates collisions within the same A/B/C group.'},
       {'role': 'user', 'content': 'After buying from Hagen, users often buy from:'},
       {'role': 'assistant', 'content': 'Cats'}], dtype=object)